# 23 — Does rubric v4 work on fresh demonstrations?

**DR-69 · Gemini 3.8 Flash and Qwen3.6 · Fixed requests, reviewed answers**

Notebook 22 improved results on examples that helped shape the rubric. This notebook checks new event types, without changing the models or prompts after seeing their answers. It is a small confirmation study, not proof of production reliability.

We send 40 original titles with the full v4 rubric. Search is off; learned knowledge is allowed. **34 resolved fresh cases** form the main comparison; **all 38 resolved cases** form a second view. The two unclear cases have no scoring answers, but remain in coverage reporting. Historical operator tags are never used.

Private inputs and saved responses are required. Public readers can use the [aggregate findings](../docs/results/fresh-v4-confirmation.md). The notebook defaults to replay: no API calls.


In [ ]:
import copy
import json
import os
import sys
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from threading import Event

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path[:0] = [str(root), str(root / "src")]
from dotenv import load_dotenv
from notebooks.tools import verify_sha256, show_table, fingerprint, percent
from notebooks.tools.hashes import file_sha256
from notebooks.tools.hosted import chat_request
from notebooks.tools.gem_profile import gem_request, PRICING
from notebooks.tools.gemini import decode_result
from notebooks.tools.costs import BudgetGuard, cost_microusd
from demo_logic.notebook_support import save_private_json
from demo_logic.evaluation import multilabel_metrics
from demo_logic.taxonomy import topic_tags

load_dotenv(root / ".env", override=False)
LIVE = os.getenv("NOTEBOOK23_RUN_LIVE", "false").lower() == "true"

config_path = root / "notebooks/prompts/fresh-v4-confirmation-v1.json"
config = json.loads(config_path.read_text())
folder = root / "reports/fresh-confirmation/2026-10-07/comparison-v1"
run_id = config["version"]

def pinned(item):
    """Load an exact reviewed input; refuse missing private data or changed bytes."""
    path = root / item["path"]
    if not path.is_file():
        raise FileNotFoundError(f"Private input missing: {item['path']}. Use the public result summary.")
    verify_sha256(path, item["sha256"])
    return path.read_text()

reference = json.loads(pinned(config["reference"]))
cases = json.loads(pinned(config["cases"]))
baseline = json.loads(pinned(config["baseline_config"]))
rubric = pinned(config["rubric"])
assert config["candidates"] == baseline["candidates"] and config["wrapper"] == baseline["wrapper"]
reviewed = {r["case_id"]: r for r in reference["records"]}
source = {r["case_id"]: r for r in cases}
ids = sorted(reviewed)
assert set(ids) == set(source) and len(ids) == 40
assert all(reviewed[i]["title"] == source[i]["title"] for i in ids)
cohorts = {"fresh": reference["primary_fresh_case_ids"], "all_resolved": reference["all_resolved_case_ids"]}
assert len(cohorts["fresh"]) == 34 and len(cohorts["all_resolved"]) == 38
assert set(reference["unresolved_case_ids"]) == {i for i in ids if reviewed[i]["reference_tags"] is None}
cohorts["context_needed"] = [i for i in cohorts["all_resolved"] if not reviewed[i]["title_sufficient"]]
cohorts["title_sufficient"] = [i for i in cohorts["all_resolved"] if reviewed[i]["title_sufficient"]]
tags = sorted(topic_tags("v2"))
models = [c["model"] for c in config["candidates"]]
show_table(["Scoring group", "Cases"], [(k, len(v)) for k, v in cohorts.items()] + [("Unclear: no correctness score", 2)])


## 1. Fixed inputs and spending limit

Both models receive exactly the same German title and complete v4 rubric used by Notebook 22. They do not receive reference tags, review notes, translations or separate location/date fields. There are 80 scheduled requests: one per model and case.

The main targets remain **80% precision, 80% recall and 95% coverage**. No new threshold or prompt is selected here. Complete reviewer references retain third and fourth tags; the two-tag model cap limits possible recall to **92.2%** on the fresh subset.

The paid run has a **$2 ceiling**, within the shared $50 project and $1/event guards. Qwen uses the confirmed free hosted Hetzner account. Before starting, we check a conservative whole-batch allowance; each paid request is reserved in the shared ledger. Dollar estimates are separate from the euro billing dashboard.


In [ ]:
payloads = {}
for candidate in config["candidates"]:
    model = candidate["model"]
    payloads[model] = {}
    for i in ids:
        system = config["wrapper"].replace("RUBRIC_VERSION", "4") + rubric
        title = json.dumps({"title": source[i]["title"]}, ensure_ascii=False)
        p = copy.deepcopy(candidate["settings"])
        if "Qwen" in model:
            p["messages"] = [{"role": "system", "content": system}, {"role": "user", "content": title}]
        else:
            p.update(system_instruction=system, input=title)
        assert not p.get("tools")
        payloads[model][i] = p
assert PRICING["input"] == config["gemini_prices_usd_per_million"]["input"]
assert PRICING["output"] == config["gemini_prices_usd_per_million"]["output"]
gemini = next(m for m in models if m.startswith("gemini"))
bounds = {i: cost_microusd(len(json.dumps(p).encode()) + 2048,
    p["generation_config"]["max_output_tokens"], PRICING["input"], PRICING["output"])
    for i, p in payloads[gemini].items()}
batch_bound = sum(bounds.values())
assert batch_bound <= config["run_limit_usd"] * 1_000_000
contract = {"config_sha256": file_sha256(config_path), "config": config, "ids": ids,
    "cohorts": cohorts, "paid_batch_reservation_microusd": batch_bound,
    "request_fingerprints": {c["model"]: {i: fingerprint({"endpoint": c["endpoint"], "payload": p})
        for i, p in payloads[c["model"]].items()} for c in config["candidates"]}}
contract_path = folder / "contract.json"
if contract_path.exists():
    assert json.loads(contract_path.read_text()) == contract, "Inputs changed: use a new reviewed run version."
else:
    save_private_json(contract_path, contract)
print(f"Conservative paid-batch allowance: ${batch_bound / 1_000_000:.4f}; ceiling $2.00. Not an invoice.")


## 2. Run once, or replay

Leave `NOTEBOOK23_RUN_LIVE=false` (the default) to replay. Only set it true to fill missing requests for this approved comparison. The existing private `.env` supplies the provider keys. A saved invalid answer is a failure, not an invitation to keep sampling.

An operational or accounting problem pauses that provider. Original response times and usage remain saved; replay time is not inference latency. A stopped or missing batch cannot produce a completed comparison.


In [ ]:
guard = BudgetGuard(root / "reports/budget/capstone.sqlite", prior_total_microusd=0,
    prior_research_microusd=0, run_limits={run_id: config["run_limit_usd"] * 1_000_000})

def run_model(candidate):
    """Run each case once or replay its exact response, retaining failures and costs.

    Stop new calls for this provider on transport or accounting problems. Never
    change the prompt or resample invalid output to improve the measured score.
    """
    model = candidate["model"]
    answers = {}
    stopped = False
    for index, i in enumerate(ids):
        place = folder / ("qwen" if "Qwen" in model else "gemini") / i
        p = payloads[model][i]
        identity = contract["request_fingerprints"][model][i]
        if "Qwen" in model:
            failed = list((place / "hosted-failures" / identity).glob("*.json"))
            for start in (place / "hosted-attempts").glob("*-started.json"):
                if not start.with_name(start.name.replace("-started", "-finished")).exists():
                    raise RuntimeError("Interrupted Qwen attempt: inspect before resuming.")
            result = chat_request({**candidate, "key_env": "HETZNER_API_KEY"}, p, tags=tags,
                cache_dir=place / "cache", allow_live=LIVE and not stopped and not failed,
                api_key=os.getenv("HETZNER_API_KEY"), max_retry_wait_seconds=0)
        else:
            completed = sorted((place / "attempts").glob("*-finished.json"))
            if completed:
                result = json.loads(completed[-1].read_text())
                assert result["request_sha256"] == identity
                if result["status"] == "ok":
                    result.update(decode_result(result["response"], tags))
            else:
                result = gem_request(p, event_id=source[i]["external_id"], folder=place,
                    key=os.getenv("GEMINI_API_KEY"), tags=tags, guard=guard,
                    live=LIVE and not stopped, decoder=decode_result, run_id=run_id, research_budget=False)
        if result.get("status") == "failed" and result.get("error") == "Invalid or incomplete model response":
            result = {**result, "transport_status": "failed", "status": "invalid_response", "predicted": []}
        if result.get("accounting_pending") or result["status"] not in {"ok", "invalid_response"}:
            stopped = True
        answers[i] = {k: v for k, v in result.items() if k != "cached"}
        print(model, index + 1, "/40", result["status"], flush=True)
    return model, answers

with ThreadPoolExecutor(max_workers=2) as pool:
    answers = dict(pool.map(run_model, config["candidates"]))
save_private_json(folder / "progress.json", {"contract": contract, "answers": answers})
complete = all(x["status"] in {"ok", "invalid_response"} and not x.get("accounting_pending")
    for model in models for x in answers[model].values())
if not complete:
    raise RuntimeError("Batch paused or missing responses; inspect private progress before reporting final scores.")


## 3. Quality on fresh and reviewed cases

**Precision** measures how many suggestions match the reviewer reference. **Recall** measures how many reference tags were found. **F1** balances both. **Coverage** means an event received at least one tag, even if wrong. **Exact match** requires the complete reference set.

The fresh subset is the main check. The broader 38-case score includes familiar cases. Context-assisted references had information the models did not receive: compare that subset separately, without assuming a difference was caused by search. Nine cases needed extra context; 29 were marked title-sufficient.


In [ ]:
metrics = {model: {name: multilabel_metrics(
    [reviewed[i]["reference_tags"] for i in subset],
    [answers[model][i].get("predicted", []) for i in subset], tags)
    for name, subset in cohorts.items()} for model in models}
show_table(["Model", "Group", "N", "Precision", "Recall", "F1", "Coverage", "Exact match"],
    [(m, g, metrics[m][g]["records"], *[percent(metrics[m][g][k]) for k in
        ["micro_precision", "micro_recall", "micro_f1", "coverage", "exact_match"]])
     for m in models for g in cohorts])
for m in models:
    x = metrics[m]["fresh"]
    passed = all(x[k] is not None and x[k] >= goal for k, goal in
        [("micro_precision", .8), ("micro_recall", .8), ("coverage", .95)])
    print(m, "Meets all three quality targets on the fresh subset:", passed)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

fig, ax = plt.subplots(figsize=(9, 4))
measures = ["micro_precision", "micro_recall", "micro_f1", "coverage"]
x = np.arange(4)
for offset, model, color in zip([-.18, .18], models, ["#52677d", "#18a999"]):
    values = [100 * (metrics[model]["fresh"][k] or 0) for k in measures]
    bars = ax.bar(x + offset, values, .34, label="Qwen3.6" if "Qwen" in model else "Gemini 3.8 Flash", color=color)
    ax.bar_label(bars, fmt="%.1f", padding=2)
ax.scatter([0, 1, 3], [80, 80, 95], marker="_", s=400, color="#cc5933", label="Targets", zorder=3)
ax.set_xticks(x, ["Precision", "Recall", "F1", "Coverage"])
ax.set_ylim(0, 112); ax.set_ylabel("Percent"); ax.set_title("Fixed rubric v4 — 34 fresh reviewed cases")
ax.legend(loc="lower right"); fig.tight_layout(); display(fig); plt.close(fig)


## 4. Missing tags, extra tags and known limits

Per-tag counts show where each model disagrees. Sparse or absent categories cannot support a strong claim about that tag. F30/F36 intentionally omit politics despite the broader written v4 rule: a politics suggestion there is a policy disagreement, not automatically a model reasoning error. These cases remain visible in the broader score.

Four complete references exceed the output cap: three have three tags and one has four. A perfect two-tag selection can recover at most 65/70 reference tags (92.9%) across all resolved cases. Keep the reference complete rather than removing answers to improve recall.

The private comparison contains all 40 titles, English reading aids, references, predictions, missing/extra tags and context/interpretation flags. Unclear cases have no correctness verdict. The listed explanations are review prompts, not claims about a model's hidden reasoning.


In [ ]:
for model in models:
    print(model)
    show_table(["Tag", "Reference count", "Correct", "Missing", "Extra"],
        [(t, *[metrics[model]["fresh"]["per_tag"][t][k] for k in ["support", "tp", "fn", "fp"]]) for t in tags])
for model in models:
    show_table(["Policy exception", "Reference", "Suggestion"],
        [(i, "; ".join(reviewed[i]["reference_tags"]), "; ".join(answers[model][i].get("predicted", [])))
         for i in ["F30", "F36"]])


## 5. Save results, timings and costs

The report preserves all requests and responses privately. Missing/invalid outputs count as no tags; successful empty answers are reported separately. The euro dashboard is an invoice view, while the USD ledger uses conservative estimates. No exchange-rate assumption combines them.

Read the conclusion in section 6, then inspect the private comparison only where a disagreement needs explanation. Do not tune or change answers after seeing these outputs and still call the same cases untouched confirmation.


In [ ]:
from statistics import median

operations = {}
comparison = []
for model in models:
    results = [answers[model][i] for i in ids]
    durations = [r["duration_seconds"] for r in results if r.get("duration_seconds") is not None]
    costs = [r.get("gross_upper_microusd") for r in results]
    total = 0 if "Qwen" in model else sum(costs) if all(c is not None for c in costs) else None
    operations[model] = {"statuses": dict(Counter(r["status"] for r in results)),
        "median_seconds": median(durations) if durations else None,
        "p95_seconds": float(np.percentile(durations, 95)) if durations else None,
        "maximum_seconds": max(durations) if durations else None,
        "total_request_seconds": sum(durations), "gross_upper_microusd": total,
        "gross_upper_usd_per_event": total / 1_000_000 / len(ids) if total is not None else None,
        "raw_coverage_all_40": sum(bool(r.get("predicted")) for r in results) / len(ids),
        "valid_empty_answers": sum(r["status"] == "ok" and not r.get("predicted") for r in results)}
for i in ids:
    row = {"case_id": i, "title": source[i]["title"], "english": source[i].get("english", source[i].get("title_en", "")),
        "reference": reviewed[i]["reference_tags"], "fresh": i in cohorts["fresh"],
        "title_sufficient": reviewed[i]["title_sufficient"], "limitations": reviewed[i]["limitations"], "models": {}}
    for model in models:
        pred = answers[model][i].get("predicted", [])
        truth = reviewed[i]["reference_tags"]
        hints = []
        if truth is None: hints.append("Unclear human answer: no correctness verdict")
        if i in ["F30", "F36"]: hints.append("Known politics-policy difference")
        if not reviewed[i]["title_sufficient"]: hints.append("Human used additional context/background")
        if truth and len(truth) > 2: hints.append("Complete reference exceeds the two-tag output cap")
        if not hints: hints.append("Inspect meaning and rubric; disagreement alone does not identify the cause")
        row["models"][model] = {"status": answers[model][i]["status"], "predicted": pred,
            "missing": sorted(set(truth) - set(pred)) if truth is not None else None,
            "extra": sorted(set(pred) - set(truth)) if truth is not None else None,
            "review_hints": hints}
    comparison.append(row)
save_private_json(folder / "comparison.json", comparison)
save_private_json(folder / "results.json", {"contract": contract, "answers": answers,
    "metrics": metrics, "operations": operations})
show_table(["Model", "Valid / 40", "Median seconds", "P95 seconds", "Estimated USD/event", "Raw coverage / 40"],
    [(m, operations[m]["statuses"].get("ok", 0), round(operations[m]["median_seconds"], 2),
      round(operations[m]["p95_seconds"], 2), operations[m]["gross_upper_usd_per_event"],
      percent(operations[m]["raw_coverage_all_40"])) for m in models])
print("Saved private results and comparison. Clear notebook outputs before committing.")

# A readable private review file accompanies the machine-readable comparison.
def md_text(value):
    """Keep source text as plain Markdown text, escaping table and HTML syntax."""
    import html
    return html.escape(str(value)).replace("|", "\\|").replace("\n", " ")

lines = ["# DR-69 — Fresh-case comparison", "",
    "38 resolved references; main fresh subset 34. Two unclear cases have no correctness score.",
    "Search off; learned knowledge allowed; unchanged full v4 rubric. Private: do not commit.", ""]
for row in comparison:
    lines += [f"## {row['case_id']}", "", "> " + md_text(row["title"]), "",
        "**English:** " + md_text(row["english"]), "",
        "**Reference:** " + ("; ".join(row["reference"]) if row["reference"] is not None else "Unclear — not scored"),
        "", f"**Primary fresh subset:** {row['fresh']} · **Title sufficient:** {row['title_sufficient']}", ""]
    for model, result in row["models"].items():
        lines += [f"**{model}** — {result['status']}",
            "- Suggested: " + ("; ".join(result["predicted"]) or "(none)"),
            "- Missing: " + ("not scored" if result["missing"] is None else "; ".join(result["missing"]) or "(none)"),
            "- Extra: " + ("not scored" if result["extra"] is None else "; ".join(result["extra"]) or "(none)"),
            "- Review hints: " + "; ".join(result["review_hints"]), ""]
    lines += ["**Retained limitations:** " + ("; ".join(row["limitations"]) or "None specifically recorded; agreement is still with one human reference."), ""]
comparison_path = folder / "comparison.md"
comparison_path.write_text("\n".join(lines))
comparison_path.chmod(0o600)
print("Private inspection file:", comparison_path.relative_to(root))


## 6. What we learned and what comes next

**Both models meet the precision and coverage targets on the 34 fresh cases, but neither reaches 80% recall.** Gemini 3.8 Flash scores **87.5% precision, 76.6% recall, 81.7% F1 and 97.1% coverage**. Qwen3.6 scores **87.0%, 73.4%, 79.7% and 97.1%**. Gemini finds two more reference tags with the same seven extra suggestions. That is a small descriptive lead, not proof that it is always better.

**The remaining gap is incomplete agreement with the reviewer reference.** Gemini misses 8/19 human-rights reference tags and 3/6 culture tags; Qwen misses 5/19 and 4/6. Both find all six environment references, without extra environment suggestions. Gemini also finds all eleven international-politics references. These small counts do not certify category-level performance; LGBTQ has no reference examples here. The follow-up audit found shared omissions involving historical-remembrance boundaries, event format versus purpose, weak event-specific evidence and the output cap. Do not count every disagreement as a confirmed model mistake or proof of reviewer bias.

**Context and reference limits matter.** Both models abstain on one resolved, three-tag case whose reference uses earlier-event/organizer background. Those three misses stay in the score. Across nine context-needed cases, Gemini recall is 60.0% and Qwen recall is 65.0%; across the 29 title-sufficient cases, it is 84.0% and 78.0%. Different cases and unequal information access prevent attributing this gap to search alone. The known politics exceptions also produce disagreement in the broader comparison: both models suggest politics on both cases, following the broader written rule.

**The output cap explains only part of the gap.** At least five reference tags cannot be returned under a two-tag cap, but the fresh comparison has 15 Gemini omissions and 17 Qwen omissions. The cap alone cannot account for all missing tags. Complete reference sets match on only **20/34 Gemini cases and 17/34 Qwen cases**.

**Next step: take these findings to DR-27.** Recommend considering a small, mandatory-human-review prototype, with Gemini as the provisional quality candidate and Qwen as the fast free hosted alternative. Neither has met every agreed quality target, so this is not approval for automatic tagging or a production rollout. Keep v4 and the targets fixed; do not keep tuning this confirmation sample. Any further experiment should address a specific operational gap and use a separately agreed evaluation. No new labeling is required to review DR-69.

All 80 responses were valid. Both models produced two valid empty answers, giving 95% raw coverage over all 40 cases. Gemini median request time was 7.82 seconds versus 0.97 seconds for Qwen. Conservative Gemini usage estimate: $0.842444 total, about $0.0211/event; Qwen used its confirmed free tier. These are API timings and estimates, not human time savings or invoices.

Read the [public findings](../docs/results/fresh-v4-confirmation.md) and the private `comparison.md` saved by section 5. DR-69 is ready for Human Review.

**Follow-up audit, 7 October:** [Audit report](../docs/audits/2026-10-07-fresh-v4-run-audit.md). All saved requests and scores reproduced. The usage-based Gemini estimate is $0.298692; $0.842444 is the conservative guard total. Neither is an invoice. No labels, prompts or paid calls changed during this audit.
